# 🤖 AutoStream AI — Social-to-Lead Agentic Workflow
### Machine Learning Intern Assignment | ServiceHive × Inflx
**Author:** Ashish Kumar Nayak  
**LLM:** Groq LLaMA 3 (llama3-70b-8192)

---
> ✅ **Run All cells top-to-bottom.** Everything is self-contained and will work in one go.

## 📦 Step 1 — Install Libraries

In [7]:
# Install all required libraries
!pip install -q groq langchain langchain-community langgraph \
               faiss-cpu flask python-dotenv pyngrok sentence-transformers

print('✅ All libraries installed!')

✅ All libraries installed!


## 🔑 Step 2 — API Keys & Groq Setup

In [ ]:
import os
from groq import Groq

# ── Set your Groq API key ───────────────────────────────────────────────
# Get key: https://console.groq.com

try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ Groq API key loaded from Colab Secrets')
except:
    os.environ['GROQ_API_KEY'] = "YOUR_GROQ_API_KEY"
    print('⚠️ Using manual key — replace YOUR_GROQ_API_KEY')

# ── Initialize Groq client ──────────────────────────────────────────────
groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])

# 🔥 USE UPDATED MODEL (IMPORTANT)
GROQ_MODEL = "llama-3.3-70b-versatile"

# ── Test connection ─────────────────────────────────────────────────────
print("🔄 Testing Groq connection...")

try:
    test = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "Reply with OK"}]
    )
    print("✅ Groq connected!")
    print("Test:", test.choices[0].message.content.strip())

except Exception as e:
    raise RuntimeError(f"❌ Groq API failed: {e}")

# ── Intent Classifier Prompt ────────────────────────────────────────────
INTENT_SYSTEM = (
    "You are an intent classifier.\n"
    "Classify the user message into exactly one of:\n"
    "greeting\n"
    "inquiry\n"
    "high_intent\n"
    "Return only the label."
)

# ── Intent Classifier Function ──────────────────────────────────────────
def classify_intent_llm(text: str) -> str:
    try:
        response = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": INTENT_SYSTEM},
                {"role": "user", "content": text}
            ],
            temperature=0
        )

        label = response.choices[0].message.content.strip().lower()

        if label in ["greeting", "inquiry", "high_intent"]:
            return label


        return "inquiry"

    except Exception as e:
        print("⚠️ Groq error:", e)
        return "inquiry"

print(f"✅ Groq LLaMA 3 ({GROQ_MODEL}) ready!")

⚠️ Using manual key — replace YOUR_GROQ_API_KEY
🔄 Testing Groq connection...
✅ Groq connected!
Test: OK
✅ Groq LLaMA 3 (llama-3.3-70b-versatile) ready!


## 📚 Step 3 — RAG Knowledge Base

In [9]:
KNOWLEDGE_BASE = {
    'product': 'AutoStream',
    'description': 'Automated AI-powered video editing tools for content creators.',
    'pricing': {
        'basic': {'price': '$29/month', 'videos_per_month': 10,         'resolution': '720p', 'support': 'Email (business hours)', 'ai_captions': False},
        'pro':   {'price': '$79/month', 'videos_per_month': 'Unlimited', 'resolution': '4K',  'support': '24/7 priority support', 'ai_captions': True}
    },
    'features': [
        'AI-powered automated video editing',
        'Smart scene detection and auto-cutting',
        'Auto subtitle and caption generation (Pro only)',
        'Multi-platform export: YouTube, Instagram, TikTok, Twitch',
        'Cloud storage for all projects',
        'One-click branding with logos and color schemes',
        'Background noise removal',
        'Real-time collaboration (Pro only)'
    ],
    'policies': {
        'refund':       'No refunds after 7 days of purchase.',
        'free_trial':   '7-day free trial available for all new users.',
        'support':      '24/7 support is available only on the Pro plan.',
        'cancellation': 'Cancel anytime; access continues until billing period ends.',
        'data_privacy': 'All user data is encrypted and never shared with third parties.'
    }
}

with open('knowledge_base.json', 'w') as f:
    json.dump(KNOWLEDGE_BASE, f, indent=2)
print('✅ Knowledge base saved')

✅ Knowledge base saved


In [10]:
from langchain_core.documents import Document

def build_documents(kb):
    docs = []
    for plan_name, d in kb['pricing'].items():
        text = (f'AutoStream {plan_name.upper()} PLAN:\n'
                f"Price: {d['price']}\n"
                f"Videos/month: {d['videos_per_month']}\n"
                f"Resolution: {d['resolution']}\n"
                f"Support: {d['support']}\n"
                f"AI Captions: {'Yes' if d['ai_captions'] else 'No'}")
        docs.append(Document(page_content=text, metadata={'section': 'pricing', 'plan': plan_name}))
    features_text = 'AutoStream FEATURES:\n' + '\n'.join(f'- {f}' for f in kb['features'])
    docs.append(Document(page_content=features_text, metadata={'section': 'features'}))
    for policy_name, policy_text in kb['policies'].items():
        docs.append(Document(
            page_content=f'AutoStream POLICY — {policy_name.upper()}:\n{policy_text}',
            metadata={'section': 'policy', 'type': policy_name}))
    return docs

documents = build_documents(KNOWLEDGE_BASE)
print(f'✅ Created {len(documents)} documents')

✅ Created 8 documents


In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print('⏳ Building FAISS vector store (takes ~30s first time)...')
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(documents, embeddings)
vectorstore.save_local('faiss_index')
print('✅ FAISS vector store ready!')

⏳ Building FAISS vector store (takes ~30s first time)...


/tmp/ipykernel_800/4073160142.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS vector store ready!


In [12]:
def retrieve_context(query: str, k: int = 3) -> str:
    """Retrieve top-k relevant chunks for a user query."""
    results = vectorstore.similarity_search(query, k=k)
    return '\n\n---\n\n'.join(doc.page_content for doc in results)

# Quick sanity test
print('RAG test:', retrieve_context('Pro plan price')[:80], '...')
print('✅ RAG retrieval working!')

RAG test: AutoStream PRO PLAN:
Price: $79/month
Videos/month: Unlimited
Resolution: 4K
Sup ...
✅ RAG retrieval working!


## 🧠 Step 4 — Intent Detection

In [13]:
GREETING_KEYWORDS = {'hi','hello','hey','howdy','good morning','good afternoon',
    'good evening','greetings',"what's up",'sup','yo','hiya'}
HIGH_INTENT_KEYWORDS = {
    'sign up','signup','subscribe','buy','purchase','get started','try',
    'upgrade','want to','ready','start now','join','enroll','register',
    'get the pro','checkout',"i'm in","let's go",'i want'
}

def classify_intent_keywords(text: str) -> str:
    lower = text.lower().strip()
    tokens = set(re.findall(r'\b\w+\b', lower))
    if tokens & GREETING_KEYWORDS: return 'greeting'
    for phrase in HIGH_INTENT_KEYWORDS:
        if phrase in lower: return 'high_intent'
    return 'inquiry'

def classify_intent(text: str) -> str:
    """Keyword-first classifier; falls back to Groq LLaMA 3 for uncertain cases."""
    label = classify_intent_keywords(text)
    if label == 'inquiry':
        try:
            resp = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[
                    {'role': 'system', 'content': INTENT_SYSTEM},
                    {'role': 'user',   'content': text}
                ],
                max_tokens=10,
                temperature=0
            )
            llm_label = resp.choices[0].message.content.strip().lower().replace('.', '')
            if llm_label in {'greeting', 'inquiry', 'high_intent'}:
                label = llm_label
        except Exception:
            pass   # fallback: keep keyword result
    return label

# Quick test
for s in ['Hello!', 'How much is the Pro plan?', 'I want to sign up']:
    print(f'  [{classify_intent_keywords(s):12s}] {s}')
print('✅ Intent detection ready!')

  [greeting    ] Hello!
  [inquiry     ] How much is the Pro plan?
  [high_intent ] I want to sign up
✅ Intent detection ready!


## 🛠️ Step 5 — Lead Capture Tool

In [14]:
def mock_lead_capture(name: str, email: str, platform: str) -> dict:
    """Mock CRM API — validates all fields before firing."""
    missing = [f for f, v in [('name', name), ('email', email), ('platform', platform)]
               if not v or not str(v).strip()]
    if missing:
        raise ValueError(f'Missing required fields: {", ".join(missing)}')
    if not re.match(r'^[\w.+-]+@[\w-]+\.[\w.]+$', email.strip()):
        raise ValueError(f"Invalid email format: '{email}'")
    name, email, platform = name.strip().title(), email.strip().lower(), platform.strip().title()
    print(f'\n{"="*50}')
    print('  🎯  LEAD CAPTURED SUCCESSFULLY!')
    print(f'  Name     : {name}')
    print(f'  Email    : {email}')
    print(f'  Platform : {platform}')
    print(f'{"="*50}\n')
    return {'status': 'success', 'lead': {'name': name, 'email': email, 'platform': platform}}

print('✅ Lead capture tool ready!')

✅ Lead capture tool ready!


## 🔄 Step 6 — Memory & State

In [15]:
from dataclasses import dataclass, field
from collections import deque

VALID_STATES = {'normal','collecting_name','collecting_email','collecting_platform','lead_captured'}

@dataclass
class ConversationMemory:
    max_history:   int           = 6
    history:       deque         = field(default_factory=lambda: deque(maxlen=6))
    state:         str           = 'normal'
    lead_name:     Optional[str] = None
    lead_email:    Optional[str] = None
    lead_platform: Optional[str] = None

    def add(self, role, content): self.history.append({'role': role, 'content': content})
    def get_history(self): return list(self.history)
    def set_state(self, s): assert s in VALID_STATES; self.state = s
    def lead_complete(self): return all([self.lead_name, self.lead_email, self.lead_platform])
    def reset(self):
        self.lead_name = self.lead_email = self.lead_platform = None
        self.state = 'normal'

print('✅ Memory & state management ready!')

✅ Memory & state management ready!


## 🤖 Step 7 — Core Agent Logic (LangGraph + Groq LLaMA 3)

In [16]:
from langgraph.graph import StateGraph, END
import concurrent.futures

class AgentState(TypedDict):
    messages:       List[dict]
    current_state:  str
    user_input:     str
    agent_response: str
    intent:         str
    lead_name:      Optional[str]
    lead_email:     Optional[str]
    lead_platform:  Optional[str]
    lead_captured:  bool

def groq_chat(system: str, history: list, user_msg: str, timeout: int = 20) -> str:
    """Call Groq LLaMA 3 with conversation history and a hard timeout."""
    messages = [{'role': 'system', 'content': system}]
    for m in history[-4:]:
        messages.append({'role': m['role'], 'content': m['content']})
    messages.append({'role': 'user', 'content': user_msg})

    def _call():
        resp = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            max_tokens=512,
            temperature=0.7
        )
        return resp.choices[0].message.content.strip()

    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(_call)
        try:
            return future.result(timeout=timeout)
        except concurrent.futures.TimeoutError:
            return '⚠️ Response timed out. Please try again.'
        except Exception as e:
            return f'⚠️ Groq error: {str(e)[:120]}'

def detect_intent_node(state):
    if state['current_state'] in {'collecting_name','collecting_email','collecting_platform'}:
        state['intent'] = 'lead_collection'
    else:
        state['intent'] = classify_intent(state['user_input'])
    return state

def handle_greeting_node(state):
    system = (
        'You are a friendly AI sales assistant for AutoStream — an AI-powered video editing SaaS '
        'for content creators. Greet the user warmly and invite them to ask about pricing, '
        'features, or how to get started. Keep it to 2 sentences max.'
    )
    state['agent_response'] = groq_chat(system, state['messages'], state['user_input'])
    return state

def handle_inquiry_node(state):
    context = retrieve_context(state['user_input'])
    system = (
        'You are a helpful sales assistant for AutoStream, an AI-powered video editing SaaS. '
        'Answer the user question ONLY using the context below. Be concise, friendly, and '
        'highlight value where relevant. If the answer is not in the context, say so politely.'
        f'\n\nCONTEXT:\n{context}'
    )
    state['agent_response'] = groq_chat(system, state['messages'], state['user_input'])
    return state

def start_lead_collection_node(state):
    state['current_state'] = 'collecting_name'
    state['agent_response'] = (
        "That's awesome! I'd love to get you started with AutoStream Pro. 🎉\n"
        'Could I get your **full name** first?'
    )
    return state

def collect_lead_info_node(state):
    user_input = state['user_input'].strip()
    current = state['current_state']
    if current == 'collecting_name':
        state['lead_name'] = user_input
        state['current_state'] = 'collecting_email'
        state['agent_response'] = (f'Nice to meet you, **{user_input}**! 👋\n'
                                    'What is your **email address**?')
    elif current == 'collecting_email':
        if not re.match(r'^[\w.+-]+@[\w-]+\.[\w.]+$', user_input):
            state['agent_response'] = f"'{user_input}' doesn't look valid. Please try again."
            return state
        state['lead_email'] = user_input
        state['current_state'] = 'collecting_platform'
        state['agent_response'] = ('Perfect! Which **creator platform** are you on?\n'
                                    '(YouTube, Instagram, TikTok, Twitch…)')
    elif current == 'collecting_platform':
        state['lead_platform'] = user_input
        state['current_state'] = 'lead_captured'
        try:
            mock_lead_capture(state['lead_name'], state['lead_email'], state['lead_platform'])
            state['lead_captured'] = True
            state['agent_response'] = (
                f"🎉 You're all set, **{state['lead_name']}**!\n\n"
                f"Our team will reach out to **{state['lead_email']}** shortly "
                f"to activate your AutoStream Pro trial. 🚀\n\n"
                'Is there anything else I can help with?'
            )
        except ValueError as e:
            state['agent_response'] = f'Something went wrong: {e}. Let us start again.'
            state['current_state'] = 'collecting_name'
            state['lead_name'] = state['lead_email'] = state['lead_platform'] = None
    return state

def route_intent(state) -> str:
    if state['current_state'] in {'collecting_name','collecting_email','collecting_platform'}:
        return 'collect_lead'
    return {'greeting':'greeting','high_intent':'start_lead'}.get(state['intent'], 'inquiry')

def build_agent():
    g = StateGraph(AgentState)
    g.add_node('detect_intent',  detect_intent_node)
    g.add_node('greeting',       handle_greeting_node)
    g.add_node('inquiry',        handle_inquiry_node)
    g.add_node('start_lead',     start_lead_collection_node)
    g.add_node('collect_lead',   collect_lead_info_node)
    g.set_entry_point('detect_intent')
    g.add_conditional_edges('detect_intent', route_intent,
        {'greeting':'greeting','inquiry':'inquiry',
         'start_lead':'start_lead','collect_lead':'collect_lead'})
    for node in ('greeting','inquiry','start_lead','collect_lead'):
        g.add_edge(node, END)
    return g.compile()

agent_graph = build_agent()
print('✅ LangGraph agent compiled with Groq LLaMA 3!')
print('   Nodes:', list(agent_graph.get_graph().nodes))

✅ LangGraph agent compiled with Groq LLaMA 3!
   Nodes: ['__start__', 'detect_intent', 'greeting', 'inquiry', 'start_lead', 'collect_lead', '__end__']


## 💬 Step 8 — Chat Function

In [17]:
def initial_state() -> AgentState:
    return {
        'messages': [], 'current_state': 'normal',
        'user_input': '', 'agent_response': '', 'intent': '',
        'lead_name': None, 'lead_email': None, 'lead_platform': None,
        'lead_captured': False
    }

def chat(user_input: str, session: AgentState):
    """Main chat function — call this with any user message."""
    session['user_input'] = user_input
    session['messages'].append({'role': 'user', 'content': user_input})
    session['messages'] = session['messages'][-6:]
    updated = agent_graph.invoke(session)
    reply = updated['agent_response']
    updated['messages'].append({'role': 'assistant', 'content': reply})
    return reply, updated

print('✅ Chat function ready!')
print('   Usage: reply, session = chat("Hello", initial_state())')

✅ Chat function ready!
   Usage: reply, session = chat("Hello", initial_state())


## 🎨 Step 9 — Save Chat UI

In [18]:
import os
os.makedirs('templates', exist_ok=True)

HTML = r'''<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width,initial-scale=1.0"/>
<title>AutoStream AI</title>
<script src="https://cdn.tailwindcss.com"></script>
<style>
  *{box-sizing:border-box}
  body{background:linear-gradient(135deg,#0f0f0f 0%,#1a1a2e 50%,#16213e 100%);min-height:100vh}
  #chat-box{scroll-behavior:smooth}
  .user-bubble{background:linear-gradient(135deg,#6366f1,#8b5cf6);border-radius:18px 18px 4px 18px}
  .bot-bubble{background:linear-gradient(135deg,#1e293b,#334155);border-radius:18px 18px 18px 4px;border:1px solid #334155}
  @keyframes fadeIn{from{opacity:0;transform:translateY(8px)}to{opacity:1;transform:translateY(0)}}
  .msg-in{animation:fadeIn .3s ease-out}
  @keyframes bounce{0%,80%,100%{transform:translateY(0)}40%{transform:translateY(-6px)}}
  .dot{animation:bounce 1.2s infinite;width:8px;height:8px;border-radius:50%;background:#6366f1;display:inline-block}
  .dot:nth-child(2){animation-delay:.2s}.dot:nth-child(3){animation-delay:.4s}
  textarea{resize:none}
  ::-webkit-scrollbar{width:6px}
  ::-webkit-scrollbar-thumb{background:#334155;border-radius:3px}
</style>
</head>
<body class="flex items-center justify-center p-4">
<div class="w-full max-w-2xl flex flex-col bg-gray-900/80 backdrop-blur-xl rounded-3xl shadow-2xl border border-gray-700/50 overflow-hidden" style="height:90vh">
  <div class="flex items-center gap-3 px-6 py-4 border-b border-gray-700/50 bg-gray-900/60">
    <div class="w-10 h-10 rounded-full bg-gradient-to-br from-indigo-500 to-purple-600 flex items-center justify-center text-xl">🎬</div>
    <div>
      <p class="text-white font-semibold text-sm">AutoStream AI</p>
      <p class="text-green-400 text-xs">● Powered by Groq LLaMA 3</p>
    </div>
    <button onclick="resetChat()" class="ml-auto text-gray-400 hover:text-white text-xs border border-gray-600 px-3 py-1 rounded-lg transition">New Chat</button>
  </div>
  <div id="chat-box" class="flex-1 overflow-y-auto px-4 py-6 flex flex-col gap-4">
    <div class="flex gap-3 msg-in">
      <div class="w-8 h-8 rounded-full bg-gradient-to-br from-indigo-500 to-purple-600 flex items-center justify-center text-sm">🤖</div>
      <div class="bot-bubble px-4 py-3 text-gray-100 text-sm max-w-xs leading-relaxed">
        👋 Hi! I am AutoStream AI.<br>Ask me about <b>pricing</b>, <b>features</b>, or how to <b>get started</b>!
      </div>
    </div>
  </div>
  <div id="typing" class="hidden px-6 pb-2">
    <div class="flex items-center gap-3">
      <div class="w-8 h-8 rounded-full bg-gradient-to-br from-indigo-500 to-purple-600 flex items-center justify-center text-sm">🤖</div>
      <div class="bot-bubble px-4 py-3 flex gap-1 items-center">
        <div class="dot"></div><div class="dot"></div><div class="dot"></div>
      </div>
    </div>
  </div>
  <div class="px-4 py-4 border-t border-gray-700/50 bg-gray-900/60">
    <div class="flex gap-3 items-end bg-gray-800/60 rounded-2xl px-4 py-3 border border-gray-700/50 focus-within:border-indigo-500/50 transition">
      <textarea id="input-box" rows="1" placeholder="Ask about pricing, features, or sign up..."
        class="flex-1 bg-transparent text-gray-100 text-sm placeholder-gray-500 outline-none max-h-28 leading-relaxed"
        onkeydown="handleKey(event)"></textarea>
      <button onclick="sendMessage()"
        class="w-10 h-10 rounded-xl bg-gradient-to-br from-indigo-500 to-purple-600 flex items-center justify-center text-white shadow-lg transition hover:scale-105">&#10148;</button>
    </div>
    <p class="text-gray-600 text-xs text-center mt-2">AutoStream AI · Groq LLaMA 3 · Ashish Kumar Nayak</p>
  </div>
</div>
<script>
  const chatBox=document.getElementById('chat-box');
  const inputBox=document.getElementById('input-box');
  const typing=document.getElementById('typing');
  const BASE_URL=window.location.origin;
  function scrollBottom(){chatBox.scrollTop=chatBox.scrollHeight;}
  function appendMessage(role,text){
    const isUser=role==='user';
    const wrapper=document.createElement('div');
    wrapper.className='flex gap-3 msg-in '+(isUser?'flex-row-reverse':'');
    const avatar=document.createElement('div');
    avatar.className='w-8 h-8 rounded-full flex items-center justify-center text-sm flex-shrink-0 '+(isUser?'bg-gradient-to-br from-purple-500 to-pink-500':'bg-gradient-to-br from-indigo-500 to-purple-600');
    avatar.textContent=isUser?'🧑':'🤖';
    const bubble=document.createElement('div');
    bubble.className=(isUser?'user-bubble text-white':'bot-bubble text-gray-100')+' px-4 py-3 text-sm max-w-xs leading-relaxed shadow-md whitespace-pre-wrap';
    bubble.innerHTML=text.replace(/\*\*(.*?)\*\*/g,'<b>$1</b>');
    wrapper.appendChild(avatar);wrapper.appendChild(bubble);
    chatBox.appendChild(wrapper);scrollBottom();
  }
  async function sendMessage(){
    const msg=inputBox.value.trim();
    if(!msg)return;
    appendMessage('user',msg);
    inputBox.value='';inputBox.style.height='auto';
    typing.classList.remove('hidden');scrollBottom();
    try{
      const res=await fetch(`${BASE_URL}/chat`,{
        method:'POST',headers:{'Content-Type':'application/json'},
        body:JSON.stringify({message:msg})
      });
      const data=await res.json();
      typing.classList.add('hidden');
      appendMessage('bot',data.response||data.error||'⚠️ No response');
    }catch(err){
      typing.classList.add('hidden');
      appendMessage('bot','⚠️ Backend not reachable. Is Flask running?');
    }
  }
  async function resetChat(){
    await fetch(`${BASE_URL}/reset`,{method:'POST'});
    chatBox.innerHTML='';
    appendMessage('bot','🔄 Session reset! How can I help you today?');
  }
  function handleKey(e){
    if(e.key==='Enter'&&!e.shiftKey){e.preventDefault();sendMessage();}
    inputBox.style.height='auto';
    inputBox.style.height=inputBox.scrollHeight+'px';
  }
</script>
</body></html>'''

with open('templates/index.html', 'w') as f:
    f.write(HTML)
print('✅ Chat UI saved to templates/index.html')

✅ Chat UI saved to templates/index.html


## 🌐 Step 10 — Start Flask Server

In [19]:
from flask import Flask, request, jsonify, render_template_string
import threading, concurrent.futures, time

flask_app = Flask(__name__, template_folder='templates')
flask_session = initial_state()

@flask_app.route('/chat', methods=['POST'])
def chat_endpoint():
    global flask_session
    data = request.get_json(force=True)
    msg  = data.get('message', '').strip()
    if not msg:
        return jsonify({'error': 'No message provided'}), 400

    # Run chat() in a thread with a 25s hard timeout
    # This prevents the browser from loading forever
    result = {'reply': None, 'session': None, 'error': None}

    def _run():
        try:
            r, s = chat(msg, flask_session)
            result['reply']   = r
            result['session'] = s
        except Exception as e:
            result['error'] = str(e)

    t = threading.Thread(target=_run)
    t.start()
    t.join(timeout=25)   # max 25 seconds

    if t.is_alive() or result['reply'] is None:
        err = result['error'] or 'Request timed out — Groq took too long.'
        return jsonify({'response': f'⚠️ {err}'}), 200

    flask_session = result['session']
    return jsonify({'response': result['reply']})

@flask_app.route('/reset', methods=['POST'])
def reset_endpoint():
    global flask_session
    flask_session = initial_state()
    return jsonify({'status': 'session reset'})

@flask_app.route('/')
def index():
    return render_template_string(open('templates/index.html').read())

flask_thread = threading.Thread(
    target=lambda: flask_app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False),
    daemon=True
)
flask_thread.start()
time.sleep(1)
print('✅ Flask server started on port 5000')
print('   POST /chat → { "message": "..." } → { "response": "..." }')
print('   GET  /     → Chat UI')

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


✅ Flask server started on port 5000
   POST /chat → { "message": "..." } → { "response": "..." }
   GET  /     → Chat UI


## 🚀 Step 11 — Open ngrok Tunnel
> **This is the last cell.** After it runs, copy the URL printed below and open it in your browser.

In [ ]:
from pyngrok import ngrok

# ⚠️  IMPORTANT: Replace the string below with your real ngrok auth token
# Get it free at: https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token('YOUR_NGROK_AUTH_TOKEN')   # <── REPLACE THIS

# Close any old tunnels to avoid conflicts
for tunnel in ngrok.get_tunnels():
    ngrok.disconnect(tunnel.public_url)

# Open a new public HTTPS tunnel → Flask on port 5000
public_url = ngrok.connect(5000)

print('=' * 62)
print('  ✅  EVERYTHING IS RUNNING!')
print('=' * 62)
print(f'  🌐  Open this URL in your browser:')
print(f'      {public_url}')
print()
print(f'  📬  API endpoint : {public_url}/chat')
print(f'  🔄  Reset session: {public_url}/reset')
print('=' * 62)

  ✅  EVERYTHING IS RUNNING!
  🌐  Open this URL in your browser:
      NgrokTunnel: "https://7863-34-124-172-128.ngrok-free.app" -> "http://localhost:5000"

  📬  API endpoint : NgrokTunnel: "https://7863-34-124-172-128.ngrok-free.app" -> "http://localhost:5000"/chat
  🔄  Reset session: NgrokTunnel: "https://7863-34-124-172-128.ngrok-free.app" -> "http://localhost:5000"/reset
